# D2C Skincare Subscription Analytics â€” Data Simulation

## Objective

This notebook will generate a realistic synthetic dataset for users, subscriptions, orders, and marketing spend.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Set a reproducible random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
TARGET_USERS = 60_000
SIMULATION_START = "2025-01-01"
SIMULATION_END = "2027-12-31"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data/raw"

# Realistic D2C skincare acquisition channels
ACQUISITION_CHANNELS = [
    "Referral",
    "Instagram Ads",
    "Google Ads",
    "Organic",
]

## Users Table

Generate the base users dataset with clean data (no duplicates or nulls).

In [4]:
# Generate users table
user_ids = [f"U{i:06d}" for i in range(1, TARGET_USERS + 1)]

# Generate random signup dates within simulation range
date_range = pd.date_range(start=SIMULATION_START, end=SIMULATION_END, freq="D")
signup_dates = np.random.choice(date_range, size=TARGET_USERS)

# Assign acquisition channels
acquisition_channels = np.random.choice(ACQUISITION_CHANNELS, size=TARGET_USERS)

# Assign cities (realistic D2C skincare markets)
cities = [
    "New York", "Los Angeles", "Chicago", "Houston", "Phoenix",
    "Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose",
    "Austin", "Jacksonville", "Fort Worth", "Columbus", "Charlotte",
    "San Francisco", "Indianapolis", "Seattle", "Denver", "Washington",
]
city_assignments = np.random.choice(cities, size=TARGET_USERS)

# Generate email addresses
emails = [f"user{i}@email.com" for i in range(1, TARGET_USERS + 1)]

# Create the users DataFrame
users = pd.DataFrame({
    "user_id": user_ids,
    "signup_date": signup_dates,
    "acquisition_channel": acquisition_channels,
    "city": city_assignments,
    "email": emails,
})

print("Users table generated.")

Users table generated.


In [5]:
# Validation: users table
print("=== Users Table Validation ===")
print(f"Row count: {len(users):,}")
print(f"Columns: {list(users.columns)}")
print(f"Signup date range: {users['signup_date'].min()} to {users['signup_date'].max()}")
print(f"\nAcquisition channel counts:")
print(users['acquisition_channel'].value_counts().to_string())
print(f"\nNull counts:\n{users.isnull().sum().to_string()}")
print(f"Duplicate user_ids: {users['user_id'].duplicated().sum()}")

=== Users Table Validation ===
Row count: 60,000
Columns: ['user_id', 'signup_date', 'acquisition_channel', 'city', 'email']
Signup date range: 2025-01-01 00:00:00 to 2027-12-31 00:00:00

Acquisition channel counts:
acquisition_channel
Referral         15099
Instagram Ads    15042
Google Ads       14970
Organic          14889

Null counts:
user_id                0
signup_date            0
acquisition_channel    0
city                   0
email                  0
Duplicate user_ids: 0


## Subscription Events

Generate realistic subscription lifecycle events: trial_start, plan_activated, paused, cancelled.

In [6]:
# Generate subscription events for each user
np.random.seed(RANDOM_SEED)

events = []

# Probabilities for subscription journey
ACTIVATION_RATE = 0.70  # 70% of trial users activate
PAUSE_RATE = 0.20       # 20% of activated users pause
CANCEL_RATE = 0.30      # 30% of activated users cancel

for idx, row in users.iterrows():
    user_id = row['user_id']
    signup_date = pd.Timestamp(row['signup_date'])
    
    # Trial starts on signup date
    trial_start = signup_date
    events.append({'user_id': user_id, 'event_date': trial_start, 'event_type': 'trial_start'})
    
    # Determine if user activates (70% chance)
    if np.random.random() < ACTIVATION_RATE:
        # Activation happens 1-14 days after trial start
        activation_delay = np.random.randint(1, 15)
        plan_activated = trial_start + pd.Timedelta(days=activation_delay)
        
        # Ensure activation is within simulation period
        if plan_activated <= pd.Timestamp(SIMULATION_END):
            events.append({'user_id': user_id, 'event_date': plan_activated, 'event_type': 'plan_activated'})
            
            # Check if user pauses (20% chance)
            if np.random.random() < PAUSE_RATE:
                # Pause happens 30-180 days after activation
                pause_delay = np.random.randint(30, 181)
                paused = plan_activated + pd.Timedelta(days=pause_delay)
                
                if paused <= pd.Timestamp(SIMULATION_END):
                    events.append({'user_id': user_id, 'event_date': paused, 'event_type': 'paused'})
            
            # Check if user cancels (30% chance)
            if np.random.random() < CANCEL_RATE:
                # Cancellation happens 14-365 days after activation
                cancel_delay = np.random.randint(14, 366)
                cancelled = plan_activated + pd.Timedelta(days=cancel_delay)
                
                if cancelled <= pd.Timestamp(SIMULATION_END):
                    events.append({'user_id': user_id, 'event_date': cancelled, 'event_type': 'cancelled'})

# Create subscription_events DataFrame
subscription_events = pd.DataFrame(events)
subscription_events['event_date'] = pd.to_datetime(subscription_events['event_date'])

print(f"Subscription events generated: {len(subscription_events):,}")

Subscription events generated: 119,325


In [7]:
# Validation: subscription events
print("=== Subscription Events Validation ===")
print(f"Total events: {len(subscription_events):,}")
print(f"Columns: {list(subscription_events.columns)}")
print(f"\nEvent type counts:")
print(subscription_events['event_type'].value_counts().to_string())

# Check for orphan user_ids
orphans = set(subscription_events['user_id']) - set(users['user_id'])
print(f"Orphan user_ids: {len(orphans)}")

# Check date range
print(f"Event date range: {subscription_events['event_date'].min()} to {subscription_events['event_date'].max()}")

# Check for null event_types
print(f"Null event_types: {subscription_events['event_type'].isnull().sum()}")

# Verify logical ordering: trial_start should come before plan_activated
trial_dates = subscription_events[subscription_events['event_type'] == 'trial_start'].set_index('user_id')['event_date']
activation_dates = subscription_events[subscription_events['event_type'] == 'plan_activated'].set_index('user_id')['event_date']
common_users = trial_dates.index.intersection(activation_dates.index)
invalid_order = (activation_dates[common_users] < trial_dates[common_users]).sum()
print(f"Invalid trial->activation order: {invalid_order}")

=== Subscription Events Validation ===
Total events: 119,325
Columns: ['user_id', 'event_date', 'event_type']

Event type counts:
event_type
trial_start       60000
plan_activated    41542
cancelled         10311
paused             7472
Orphan user_ids: 0
Event date range: 2025-01-01 00:00:00 to 2027-12-31 00:00:00
Null event_types: 0
Invalid trial->activation order: 0


## Orders

Generate realistic D2C skincare subscription orders for activated users.

In [8]:
import pandas as pd
import numpy as np

# Generate orders for activated users
np.random.seed(RANDOM_SEED)

# Get activated users and their activation dates
activated_users = subscription_events[subscription_events['event_type'] == 'plan_activated'][['user_id', 'event_date']].copy()
activated_users.columns = ['user_id', 'activation_date']

# Plan types and their typical order values (INR)
plan_types = ['monthly', 'quarterly', 'annual']
plan_weights = [0.60, 0.25, 0.15]  # Monthly most common

# Order value ranges by plan type (INR)
value_ranges = {
    'monthly': (499, 999),
    'quarterly': (1299, 2499),
    'annual': (3999, 7999),
}

# Status distribution
statuses = ['completed', 'failed', 'refunded']
status_weights = [0.88, 0.08, 0.04]

orders = []
order_counter = 1

for _, row in activated_users.iterrows():
    user_id = row['user_id']
    activation_date = row['activation_date']
    
    # Determine if user makes repeat purchases (60% chance)
    if np.random.random() < 0.60:
        # Number of orders: 1-6
        num_orders = np.random.randint(1, 7)
    else:
        num_orders = 1
    
    for i in range(num_orders):
        # Order date: activation + 0-365 days
        order_delay = np.random.randint(0, 366)
        order_date = activation_date + pd.Timedelta(days=order_delay)
        
        # Ensure order date is within simulation period
        if order_date > pd.Timestamp(SIMULATION_END):
            continue
        
        # Select plan type
        plan_type = np.random.choice(plan_types, p=plan_weights)
        
        # Generate order value within range
        min_val, max_val = value_ranges[plan_type]
        order_value = round(np.random.uniform(min_val, max_val), 2)
        
        # Select status
        status = np.random.choice(statuses, p=status_weights)
        
        # Create order ID
        order_id = f"ORD{order_counter:07d}"
        order_counter += 1
        
        orders.append({
            'order_id': order_id,
            'user_id': user_id,
            'order_date': order_date,
            'order_value': order_value,
            'plan_type': plan_type,
            'status': status,
        })

# Create orders DataFrame
orders = pd.DataFrame(orders)
orders['order_date'] = pd.to_datetime(orders['order_date'])

print(f"Orders generated: {len(orders):,}")

Orders generated: 86,364


In [9]:
# Validation: orders
print("=== Orders Validation ===")
print(f"Row count: {len(orders):,}")
print(f"Columns: {list(orders.columns)}")
print(f"Order date range: {orders['order_date'].min()} to {orders['order_date'].max()}")
print(f"\nOrder value summary:")
print(orders['order_value'].describe().to_string())
print(f"\nOrder value <= 0 count: {(orders['order_value'] <= 0).sum()}")
print(f"\nStatus counts:")
print(orders['status'].value_counts().to_string())
print(f"\nPlan type counts:")
print(orders['plan_type'].value_counts().to_string())
print(f"\nNull counts:")
print(orders.isnull().sum().to_string())

orphans = set(orders['user_id']) - set(users['user_id'])
print(f"\nOrphan user_ids: {len(orphans)}")

=== Orders Validation ===
Row count: 86,364
Columns: ['order_id', 'user_id', 'order_date', 'order_value', 'plan_type', 'status']
Order date range: 2025-01-09 00:00:00 to 2027-12-31 00:00:00

Order value summary:
count    86364.000000
mean      1832.721508
std       1897.360777
min        499.000000
25%        707.937500
50%        915.710000
75%       2022.440000
max       7998.900000

Order value <= 0 count: 0

Status counts:
status
completed    76108
failed        6800
refunded      3456

Plan type counts:
plan_type
monthly      51936
quarterly    21273
annual       13155

Null counts:
order_id       0
user_id        0
order_date     0
order_value    0
plan_type      0
status         0

Orphan user_ids: 0


## Marketing Spend

Generate monthly marketing spend data by acquisition channel for 2025-01 through 2027-12.

In [10]:
# Generate marketing spend data
np.random.seed(RANDOM_SEED)

# Create monthly date range for the simulation period
months = pd.date_range(start=SIMULATION_START, end=SIMULATION_END, freq='MS')

marketing_data = []

# Base spend ranges by canonical channel (INR per month)
channel_spend_ranges = {
    'Referral': (30000, 100000),
    'Instagram Ads': (200000, 500000),
    'Google Ads': (150000, 350000),
    'Organic': (20000, 80000),
}

# Base new users acquired per canonical channel per month
channel_user_ranges = {
    'Referral': (150, 700),
    'Instagram Ads': (1000, 3500),
    'Google Ads': (800, 2500),
    'Organic': (300, 1200),
}

for month in months:
    for channel in ACQUISITION_CHANNELS:
        # Generate spend with some random variation
        min_spend, max_spend = channel_spend_ranges[channel]
        spend_inr = round(np.random.uniform(min_spend, max_spend), 2)
        
        # Generate new users acquired
        min_users, max_users = channel_user_ranges[channel]
        new_users_acquired = np.random.randint(min_users, max_users + 1)
        
        marketing_data.append({
            'month': month,
            'acquisition_channel': channel,
            'spend_inr': spend_inr,
            'new_users_acquired': new_users_acquired,
        })

# Create marketing_spend DataFrame
marketing_spend = pd.DataFrame(marketing_data)
marketing_spend['month'] = pd.to_datetime(marketing_spend['month'])

print(f"Marketing spend records generated: {len(marketing_spend):,}")

Marketing spend records generated: 144


In [11]:
# Validation: marketing spend
print("=== Marketing Spend Validation ===")
print(f"Row count: {len(marketing_spend):,}")
print(f"Columns: {list(marketing_spend.columns)}")
print(f"Month range: {marketing_spend['month'].min()} to {marketing_spend['month'].max()}")
print(f"\nNull counts:")
print(marketing_spend.isnull().sum().to_string())
print(f"\nCanonical channels: {ACQUISITION_CHANNELS}")
print(f"Unexpected channels: {sorted(set(marketing_spend['acquisition_channel']) - set(ACQUISITION_CHANNELS))}")
print(f"Channel counts:")
print(marketing_spend['acquisition_channel'].value_counts().to_string())
print(f"\nSpend summary (INR):")
print(marketing_spend['spend_inr'].describe().to_string())
print(f"\nNew users summary:")
print(marketing_spend['new_users_acquired'].describe().to_string())
print(f"\nSpend < 0 count: {(marketing_spend['spend_inr'] < 0).sum()}")
print(f"New users < 0 count: {(marketing_spend['new_users_acquired'] < 0).sum()}")

=== Marketing Spend Validation ===
Row count: 144
Columns: ['month', 'acquisition_channel', 'spend_inr', 'new_users_acquired']
Month range: 2025-01-01 00:00:00 to 2027-12-01 00:00:00

Null counts:
month                  0
acquisition_channel    0
spend_inr              0
new_users_acquired     0

Canonical channels: ['Referral', 'Instagram Ads', 'Google Ads', 'Organic']
Unexpected channels: []
Channel counts:
acquisition_channel
Referral         36
Instagram Ads    36
Google Ads       36
Organic          36

Spend summary (INR):
count       144.000000
mean     180031.947639
std      141616.842427
min       20795.900000
25%       55430.130000
50%      125434.035000
75%      290771.010000
max      492126.660000

New users summary:
count     144.000000
mean     1348.888889
std       875.578076
min       182.000000
25%       625.750000
50%      1120.000000
75%      1999.500000
max      3454.000000

Spend < 0 count: 0
New users < 0 count: 0


## Business Signals

Add realistic, intentional differences by acquisition channel:
- funnel leak (low activation)
- worse retention (higher cancellation)
- weak LTV:CAC (low order value)

In [12]:
# Add deterministic business signals by canonical channel
np.random.seed(RANDOM_SEED)

# Map the original signal intent onto the canonical channels
FUNNEL_LEAK_CHANNEL = 'Organic'
WORSE_RETENTION_CHANNEL = 'Instagram Ads'
WEAK_LTV_CAC_CHANNEL = 'Referral'

# 1. FUNNEL LEAK: Reduce activation rate for Organic
organic_users = users[users['acquisition_channel'] == FUNNEL_LEAK_CHANNEL]['user_id'].values
organic_activations = subscription_events[
    (subscription_events['user_id'].isin(organic_users)) &
    (subscription_events['event_type'] == 'plan_activated')
]
activations_to_remove = organic_activations.sample(frac=0.60, random_state=RANDOM_SEED)
subscription_events.drop(activations_to_remove.index, inplace=True)

# 2. WORSE RETENTION: Increase cancellation rate for Instagram Ads
instagram_users = users[users['acquisition_channel'] == WORSE_RETENTION_CHANNEL]['user_id'].values
instagram_activations = subscription_events[
    (subscription_events['user_id'].isin(instagram_users)) &
    (subscription_events['event_type'] == 'plan_activated')
]
instagram_cancelled = subscription_events[
    (subscription_events['user_id'].isin(instagram_users)) &
    (subscription_events['event_type'] == 'cancelled')
]['user_id'].unique()
instagram_no_cancel = instagram_activations[~instagram_activations['user_id'].isin(instagram_cancelled)]
extra_cancellations = instagram_no_cancel.sample(frac=0.50, random_state=RANDOM_SEED).copy()
extra_cancellations['event_type'] = 'cancelled'
extra_cancellations['event_date'] = extra_cancellations['event_date'] + pd.Timedelta(days=60)
subscription_events = pd.concat([subscription_events, extra_cancellations], ignore_index=True)

# 3. WEAK LTV:CAC: Reduce order values for Referral
referral_users = users[users['acquisition_channel'] == WEAK_LTV_CAC_CHANNEL]['user_id'].values
referral_mask = orders['user_id'].isin(referral_users)
orders.loc[referral_mask, 'order_value'] = orders.loc[referral_mask, 'order_value'] * 0.45

print('Business signals applied:')
print(f'  Funnel leak: {FUNNEL_LEAK_CHANNEL} (removed {len(activations_to_remove)} activations)')
print(f'  Worse retention: {WORSE_RETENTION_CHANNEL} (added {len(extra_cancellations)} cancellations)')
print(f'  Weak LTV:CAC: {WEAK_LTV_CAC_CHANNEL} (reduced order values by 55%)')

Business signals applied:
  Funnel leak: Organic (removed 6199 activations)
  Worse retention: Instagram Ads (added 3932 cancellations)
  Weak LTV:CAC: Referral (reduced order values by 55%)


In [13]:
# Validation: business signals
print('=== Business Signals Validation ===')

# 1. Activation rate by channel
print('\n1. Activation Rate by Channel:')
trial_counts = users.groupby('acquisition_channel')['user_id'].count()
activation_by_channel = subscription_events[subscription_events['event_type'] == 'plan_activated']['user_id'].map(
    users.set_index('user_id')['acquisition_channel']
).value_counts()
activation_rate = (activation_by_channel / trial_counts * 100).round(1)
print(activation_rate.sort_values().to_string())

# 2. Cancellation rate by channel
print('\n2. Cancellation Rate by Channel:')
activated_by_channel = subscription_events[subscription_events['event_type'] == 'plan_activated']['user_id'].map(
    users.set_index('user_id')['acquisition_channel']
).value_counts()
cancelled_by_channel = subscription_events[subscription_events['event_type'] == 'cancelled']['user_id'].map(
    users.set_index('user_id')['acquisition_channel']
).value_counts()
cancellation_rate = (cancelled_by_channel / activated_by_channel * 100).round(1)
print(cancellation_rate.sort_values(ascending=False).to_string())

# 3. Average order value by channel
print('\n3. Average Order Value by Channel:')
orders_with_channel = orders.merge(users[['user_id', 'acquisition_channel']], on='user_id')
avg_order_value = orders_with_channel.groupby('acquisition_channel')['order_value'].mean().round(2)
print(avg_order_value.sort_values().to_string())

print('\n--- Signal Summary ---')
print(f'Funnel leak: {FUNNEL_LEAK_CHANNEL}')
print(f'Worse retention: {WORSE_RETENTION_CHANNEL}')
print(f'Weak LTV:CAC: {WEAK_LTV_CAC_CHANNEL}')


=== Business Signals Validation ===

1. Activation Rate by Channel:
user_id
Organic          27.8
Google Ads       68.5
Instagram Ads    69.5
Referral         69.6

2. Cancellation Rate by Channel:
user_id
Instagram Ads    62.4
Organic          61.8
Referral         24.9
Google Ads       24.8

3. Average Order Value by Channel:
acquisition_channel
Referral          814.55
Instagram Ads    1828.54
Organic          1841.24
Google Ads       1851.46

--- Signal Summary ---
Funnel leak: Organic
Worse retention: Instagram Ads
Weak LTV:CAC: Referral


## Messiness Injection

Inject realistic data quality issues:
- Inconsistent acquisition-channel names/casing
- ~5% null city values
- ~1% duplicate user_id rows
- Mixed event_date formats
- Some null event_type values
- Some zero/negative order_value
- Some orphan user_id values in orders
- Some missing channel-month combinations in marketing spend

In [14]:
# Messiness Injection
# Apply all intentional data-quality issues in one deterministic step

np.random.seed(RANDOM_SEED)

# ============================================================
# 1. Inconsistent acquisition channel names
# ============================================================
# The canonical channel labels are intentionally kept unchanged here.
channel_map = {}
users["acquisition_channel"] = users["acquisition_channel"].replace(channel_map)


# ============================================================
# 2. ~5% null city values
# ============================================================
city_mask = np.random.random(len(users)) < 0.05
users.loc[city_mask, "city"] = None


# ============================================================
# 3. ~1% duplicate user_id rows
# ============================================================
duplicate_count = int(len(users) * 0.01)
duplicate_rows = users.sample(
    n=duplicate_count,
    random_state=RANDOM_SEED
).copy()

users = pd.concat(
    [users, duplicate_rows],
    ignore_index=True
)


# ============================================================
# 4. Mixed event_date formats
# ============================================================
subscription_events["event_date"] = subscription_events["event_date"].astype(object)

date_mask = np.random.random(len(subscription_events)) < 0.02

subscription_events.loc[date_mask, "event_date"] = (
    pd.to_datetime(
        subscription_events.loc[date_mask, "event_date"]
    ).dt.strftime("%d/%m/%Y")
)


# ============================================================
# 5. Null event_type values
# ============================================================
event_type_mask = np.random.random(len(subscription_events)) < 0.02
subscription_events.loc[event_type_mask, "event_type"] = None


# ============================================================
# 6. Zero / negative order values
# ============================================================
order_problem_count = max(1, int(len(orders) * 0.015))

problem_order_indices = orders.sample(
    n=order_problem_count,
    random_state=RANDOM_SEED
).index

half = len(problem_order_indices) // 2

orders.loc[problem_order_indices[:half], "order_value"] = 0
orders.loc[problem_order_indices[half:], "order_value"] = -100


# ============================================================
# 7. Orphan user_ids in orders
# ============================================================
orphan_count = max(1, int(len(orders) * 0.005))

orphan_indices = orders.sample(
    n=orphan_count,
    random_state=RANDOM_SEED + 1
).index

orders.loc[orphan_indices, "user_id"] = [
    f"ORPHAN_{i:04d}"
    for i in range(1, orphan_count + 1)
]


# ============================================================
# 8. Missing channel-month combinations
# ============================================================
marketing_spend = marketing_spend.copy()

# Remove 6 deterministic channel-month rows
rows_to_remove = marketing_spend.sample(
    n=6,
    random_state=RANDOM_SEED
).index

marketing_spend = marketing_spend.drop(
    rows_to_remove
).reset_index(drop=True)


print("=== Messiness Injection Applied ===")
print(f"Users rows after duplicates: {len(users):,}")
print(f"Subscription events: {len(subscription_events):,}")
print(f"Orders: {len(orders):,}")
print(f"Marketing spend rows: {len(marketing_spend):,}")

=== Messiness Injection Applied ===
Users rows after duplicates: 60,600
Subscription events: 117,058
Orders: 86,364
Marketing spend rows: 138


In [15]:
# Validation: Messiness Injection
print("=== Messiness Injection Validation ===")

print(f"Users rows: {len(users):,}")
print(f"Duplicate user_id rows: {users['user_id'].duplicated().sum():,}")
print(f"Null city values: {users['city'].isna().sum():,}")

print("\nEvent Date Types:")
print(subscription_events["event_date"].map(type).value_counts().to_string())

print(f"\nNull event_type: {subscription_events['event_type'].isna().sum():,}")

print(f"\nOrder value <= 0: {(orders['order_value'] <= 0).sum():,}")

orphans = set(orders["user_id"]) - set(users["user_id"])
print(f"Orphan user_ids in orders: {len(orphans):,}")

expected_marketing_combinations = 36 * 4
actual_marketing_combinations = marketing_spend[["month", "acquisition_channel"]].drop_duplicates().shape[0]
print(f"\nMarketing spend rows: {len(marketing_spend):,}")
print(f"Expected marketing combinations if complete: {expected_marketing_combinations:,}")
print(f"Actual marketing combinations after intentional missing rows: {actual_marketing_combinations:,}")
print(f"Missing channel-month combinations: {expected_marketing_combinations - actual_marketing_combinations:,}")

=== Messiness Injection Validation ===
Users rows: 60,600
Duplicate user_id rows: 600
Null city values: 3,028

Event Date Types:
event_date
<class 'pandas.Timestamp'>    114745
<class 'str'>                   2313

Null event_type: 2,354

Order value <= 0: 1,295
Orphan user_ids in orders: 431

Marketing spend rows: 138
Expected marketing combinations if complete: 144
Actual marketing combinations after intentional missing rows: 138
Missing channel-month combinations: 6


In [16]:
# Export raw tables
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

raw_tables = {
    'users': users,
    'subscription_events': subscription_events,
    'orders': orders,
    'marketing_spend': marketing_spend,
}

for table_name, table in raw_tables.items():
    table.to_csv(RAW_DATA_DIR / f'{table_name}.csv', index=False)

print(f'Raw tables exported to {RAW_DATA_DIR.resolve()}')

Raw tables exported to D:\cooding_playground\data analysist projects\D2C-Skincare-Subscription-Analytics\notebooks\data\raw
